In [1]:
## Create a vector of the required packages for this analysis
req_packages <- c("edgeR", "janitor", "RUVSeq", "splines", "stringr", "tidyverse", "vegan", "viridis")

## load the packages, quietly
invisible(suppressWarnings(suppressMessages(
    lapply(req_packages, require, character.only = TRUE)
)))

### Load in Data

In [3]:
## load in raw counts
raw_reads <- read_tsv("results/abundances/SB.02.06/stringtie_gene_counts.txt", skip = 1) %>%
    select(-c(Chr, Start, End, Strand, Length))

## clean up column names
names <- colnames(raw_reads) %>%
    as.data.frame()
colnames(names) <- "name"

samples <- names %>%
    mutate(name = str_remove_all(name, "results/HISAT-2/SB.02.06/americana_"),
           name = str_remove_all(name, ".csorted.hisat2.bam")) %>%
    filter(grepl("SB", name)) %>%
    mutate(name = str_remove_all(name, "_SB.02.06")) %>%
    pull("name")
colnames <- c("gene", samples)
colnames(raw_reads) <- colnames

head(raw_reads)

Rows: 34392 Columns: 17
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr  (5): Geneid, Chr, Start, End, Strand
dbl (12): Length, results/HISAT-2/SB.02.06/americana_AG_1_SB.02.06.csorted.h...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


gene,AG_1,AG_2,AG_3,carcass_1,carcass_2,carcass_3,EB_1,EB_2,testes_1,testes_2,testes_3
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
XLOC_000001,0,0,0,0,0,0,0,0,0,0,0
XLOC_000002,0,0,0,0,0,0,0,0,0,0,0
XLOC_000003,0,0,0,0,0,0,0,0,0,0,0
XLOC_000004,0,0,0,0,0,0,0,0,0,0,0
XLOC_000005,0,0,0,0,0,0,0,0,0,0,0
XLOC_000006,0,0,0,0,0,0,0,0,0,0,0


### Calculate Tissue Differential Expression

In [4]:
raw_counts <- raw_reads %>%
    column_to_rownames("gene")

## convert to counts per million and remove reads with fewer than 5 reads per 3 samples
cpm_cm <- cpm(raw_counts)
thresh_cm <- cpm_cm > 5
keep_cm <- rowSums(thresh_cm) >= 3
reads <- raw_counts[keep_cm,]

head(reads)

,AG_1,AG_2,AG_3,carcass_1,carcass_2,carcass_3,EB_1,EB_2,testes_1,testes_2,testes_3
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
XLOC_003258,2598,3536,4238,14420,19730,8323,4585,8178,2841,3645,4233
XLOC_003259,188,208,203,419,506,366,202,425,478,717,899
XLOC_003260,811,496,536,546,546,319,5,17,228,204,307
XLOC_003261,307,242,259,742,790,490,198,395,206,193,250
XLOC_003262,113,133,132,646,1082,686,186,277,174,234,249
XLOC_003263,2,13,16,252,1109,459,166,218,27,73,46


In [5]:
## create sample information data frame
reads_names <- colnames(reads) %>%
    as.data.frame()
colnames(reads_names) <- "sample"

sample_info <- reads_names %>%
    separate_wider_delim(sample, delim = "_", names = c("organ", "replicate"), cols_remove = FALSE)

## create design matrix 
groups <- sample_info$organ
design <- model.matrix(~0 + groups)
rownames(design) <- colnames(reads)
colnames(design) <- str_replace(colnames(design), "groups", "")

head(design)

,AG,carcass,EB,testes
AG_1,1,0,0,0
AG_2,1,0,0,0
AG_3,1,0,0,0
carcass_1,0,1,0,0
carcass_2,0,1,0,0
carcass_3,0,1,0,0


#### Remove unwanted variation

In [6]:
## create DGE object
set <- newSeqExpressionSet(as.matrix(reads), phenoData = data.frame(groups, row.names = colnames(reads)))
set <- betweenLaneNormalization(set, which="upper")

y <- DGEList(counts=counts(set), group=groups)
y <- calcNormFactors(y, method="upperquartile")
y <- estimateDisp(y, design, robust = T)
fit <- glmQLFit(y, design, dispersion = y$tagwise.dispersion, robust = T)
res <- residuals(fit, type="deviance")

## normalize with RUVr
batch_ruv_res <- RUVr(set, rownames(reads), k = 3, res)

## create new design matrix with RUVr coefficients
design_2 <- model.matrix(~ 0 + groups, data = pData(batch_ruv_res))
colnames(design_2) <- gsub("groups", "", colnames(design_2))

## create DGElist object with new design
dgeList <- DGEList(counts = reads, group = groups)
dgeList <- calcNormFactors(dgeList)
dgeList <- estimateDisp(dgeList, design_2)
dgeList_fit <- glmQLFit(dgeList, design_2, robust = TRUE)
summary(dgeList$tagwise.dispersion)

## normalize with new design
y <- estimateDisp(y, design_2, robust = T)
fit <- glmQLFit(y, design_2, dispersion = y$tagwise.dispersion, robust = T)
res <- residuals(fit, type="deviance")

## normalize with RUVr
batch_ruv_res <- RUVr(set, rownames(reads), k = 3,res)
RUVrNormalizedCounts <- normCounts(object = batch_ruv_res)
rownames(RUVrNormalizedCounts) <- rownames(reads)

## reformat normalized reads for saving
normalized_reads <- RUVrNormalizedCounts %>%
    as.data.frame() %>%
    rownames_to_column("gene")
head(normalized_reads)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
0.01433 0.06328 0.10335 0.29506 0.24922 4.00862 

,gene,AG_1,AG_2,AG_3,carcass_1,carcass_2,carcass_3,EB_1,EB_2,testes_1,testes_2,testes_3
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,XLOC_003258,5787,9102,8243,12902,10662,10117,8173,10616,2552,2320,2337
2,XLOC_003259,466,446,450,349,352,358,443,480,415,445,483
3,XLOC_003260,1292,1045,1749,341,383,422,15,14,143,149,193
4,XLOC_003261,527,660,592,508,540,627,383,478,159,135,148
5,XLOC_003262,288,311,258,593,649,717,353,347,162,143,132
6,XLOC_003263,14,21,20,527,397,320,283,310,43,32,18


#### Calculate LogFC for each tissue compared to the rest

In [7]:
## create empty logfc table to add the future log fold change data to
mrt_specificity <- matrix(data = NA, nrow = 1, ncol = 4,
                          dimnames = list(c("remove"), c("gene", "logfc", "fdr", "organ"))) %>%
                   as.data.frame()

## calculate logfc for each organ compared to the rest of the tissues
for (i in unique(groups)) {

    ## reformat for a pairwise comparison
    organ_info <- sample_info %>%
        mutate(test = case_when(organ == i ~ i,
                                TRUE ~ "other"))
    organ_groups <- organ_info$test
    organ_design <- model.matrix(~0 + groups)

    ## run differential expression
    organ_dgeList <- DGEList(counts = normalized_reads, group = organ_groups)
    organ_dgeList <- calcNormFactors(organ_dgeList)
    organ_dgeList <- estimateDisp(organ_dgeList, organ_design)
    organ_dgeList_fit <- glmQLFit(organ_dgeList, organ_design, robust = TRUE)

    organ_dge <- glmTreat(organ_dgeList_fit, coef = 2)
    organ_dge_tags <- topTags(organ_dge, n = NULL)

    ## get logfc and statistical information
    organ_logfc <- organ_dge_tags$table %>%
        select(gene, logFC, FDR) %>%
        rename(logfc = logFC, fdr = FDR) %>%
        mutate(organ = i)

    ## add organ information to data frame
    mrt_specificity <- rbind(mrt_specificity, organ_logfc)

}

## remove the placeholder first line
mrt_specificity <- mrt_specificity %>%
    na.omit()

mrt_specificity$fdr_bh <- p.adjust(mrt_specificity$fdr, method = "bonferroni")

## get the genes that are tissue specific
testes_genes <- mrt_specificity %>%
    filter(organ == "testes") %>%
    filter(logfc < 4, fdr_bh < 0.0001)


## write out the created files
write_csv(mrt_specificity, "results/mrt_logfc.csv")
write_csv(testes_genes, "results/testes_logfc.csv")

Setting first column of `counts` as gene annotation.

Setting first column of `counts` as gene annotation.

Setting first column of `counts` as gene annotation.

Setting first column of `counts` as gene annotation.



### Calculate Tau

In [8]:
## create matrix
reads_mx <- raw_reads %>%
    column_to_rownames("gene") %>%
    as.matrix()
reads_mx <- reads_mx[rowSums(abs(reads_mx)) > 0, ]

## create groups information
sample_info <- colnames(reads_mx) %>%
    as.data.frame() %>%
    rename(sample = 1) %>%
    separate_wider_delim(sample, delim = "_", names = c("organ", "replicate"), cols_remove = FALSE)
groups <- sample_info$organ

## create design matrix 
design <- model.matrix(~0 + groups)
rownames(design) <- colnames(reads_mx)
colnames(design) <- str_replace(colnames(design), "groups", "")

head(design)

,AG,carcass,EB,testes
AG_1,1,0,0,0
AG_2,1,0,0,0
AG_3,1,0,0,0
carcass_1,0,1,0,0
carcass_2,0,1,0,0
carcass_3,0,1,0,0


In [9]:
## normalize count matrix with edger
dge_list <- DGEList(counts = reads_mx, group = groups)
norm_list <- calcNormFactors(dge_list, method = "upperquartile")
disp_list <- estimateCommonDisp(norm_list, design, robust = T)

## get normalized counts
norm_counts <- disp_list$pseudo.counts %>%
    as.data.frame() %>%
    rownames_to_column("gene")

## get mean and log transform normalized counts
reads_log <- norm_counts %>%
    pivot_longer(!gene, names_to = "sample", values_to = "value") %>%
    separate_wider_delim(sample, delim = "_", names = c("organ", "replicate")) %>%
    group_by(gene, organ) %>%
    reframe(mean = log(mean(value))) %>%
    mutate(mean = case_when(mean < 0 ~ 0,
                            TRUE ~ mean)) %>%
    pivot_wider(names_from = organ, values_from = mean) %>%
    column_to_rownames("gene")

head(reads_log)

,AG,EB,carcass,testes
,<dbl>,<dbl>,<dbl>,<dbl>
XLOC_003258,8.827839,9.003391,9.141251,7.682467
XLOC_003259,5.994864,5.962281,5.666548,6.026520
XLOC_003260,7.163173,2.545855,5.755118,5.027071
XLOC_003261,6.318306,5.915405,6.114493,4.905018
XLOC_003262,5.529676,5.717748,6.279657,4.891851
XLOC_003263,2.954437,5.552285,5.958744,3.354102


In [10]:
## create tau function
tau_function <- function(j){
	sum(1-reads_log[j,]/max(reads_log[j,]))/(length(reads_log[j,])-1)
	}

## apply to matrix
tau_est <- sapply(1:nrow(reads_log), tau_function)

## save tau as a data frame
tau_df <- tau_est %>%
    as.data.frame() %>%
    rename("tau" = 1) %>%
    mutate(tau = replace_na(tau, 0),
           gene = rownames(reads_log)) %>%
    select(gene, tau)
head(tau_df)

,gene,tau
,<chr>,<dbl>
1,XLOC_003258,0.06964972
2,XLOC_003259,0.02521456
3,XLOC_003260,0.37978862
4,XLOC_003261,0.10656878
5,XLOC_003262,0.14330377
6,XLOC_003263,0.33650311


In [11]:
## find the difference between tissues and assign tissue specificity
dagrp_tau <- reads_log %>%
    rownames_to_column("gene") %>%
    left_join(tau_df, by = "gene") %>%
    pivot_longer(!c(gene, tau), names_to = "sample", values_to = "value") %>%
    group_by(gene, tau) %>%
    reframe(max_organ = sample[which.max(value)],
            max = max(value),
            max2 = sort(value, decreasing = TRUE)[2],
            perc = (max-max2)/max) %>%
    select(-c(max, max2)) %>%
    mutate(perc = replace_na(perc, 0),
           tau_tissue = case_when(perc > 0.1 & tau > 0.5 ~ max_organ,
                                  TRUE ~ "ambiguous"))
head(dagrp_tau)

gene,tau,max_organ,perc,tau_tissue
<chr>,<dbl>,<chr>,<dbl>,<chr>
XLOC_003258,0.06964972,carcass,0.015081119,ambiguous
XLOC_003259,0.02521456,testes,0.005252837,ambiguous
XLOC_003260,0.37978862,AG,0.196568514,ambiguous
XLOC_003261,0.10656878,AG,0.032257572,ambiguous
XLOC_003262,0.14330377,carcass,0.089480798,ambiguous
XLOC_003263,0.33650311,carcass,0.068212179,ambiguous


In [12]:
## find the percent of genes in each tissue specific
dagrp_tau %>%
    group_by(tau_tissue) %>%
    reframe(count = n()) %>%
    mutate(percent = (count/sum(count))*100)

tau_tissue,count,percent
<chr>,<int>,<dbl>
AG,489,3.532216
EB,213,1.538573
ambiguous,9582,69.214100
carcass,1835,13.254840
testes,1725,12.460272


In [13]:
## write csv
write_csv(dagrp_tau, "results/mrt_tau_0.50.csv")